In [ ]:
# Imports and figure defaults.
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch
from IPython.display import display

# Keep text editable in PDF and SVG exports.
plt.rcParams.update({
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'svg.fonttype': 'none',
    'font.size': 10,
})


In [ ]:
# Input paths, metric directions, and plotting options.
DATA_PATH = Path(
    '/home/liyuanzhe/lyz32/3DCell-master/tissue_aware_confocal_metrics_final.csv'
)
SAVE_PATH = Path('./figure_output/global_metric_landscape/')
AP_COLUMN = 'AP'  # Plotted AP uses the rule-based score.

TOP_K = 5
LABEL_TOP_METHODS = False
SHOW_SCATTER_LEGEND = True
SHOW_FIGURES = True
SCATTER_XLIM = (0.49, 1.01)  # Set to None for automatic limits.
FIGSIZE = (7.6, 7.2)
SCATTER_FIGSIZE = (9.0, 7.2)
DPI = 300
EXPORT_FORMATS = ('pdf', 'svg', 'png')

RI_METRICS = ['FR', 'OSR', 'USR', 'MNR', 'IoU', 'AP']
HIGHER_IS_BETTER = {'FR', 'IoU', 'AP'}
LOWER_IS_BETTER = {'OSR', 'USR', 'MNR'}

# Include only these methods; ignore any additional methods in the CSV.
METHOD_LIST = [
    'Cellpose-SAM', 'StarDist-3D', 'Cellpose', 'PlantSeg', 'CellSAM',
    'u-Segment3D', 'Omnipose', 'CellProfiler', 'Ilastik', 'uSAM',
    'Cellpose-2D', 'Mesmer', 'Swin-UNETR', 'CellSeg3D',
]
METHOD_COLORS = {
    '3D U-Net': '#0055b3', 'StarDist-3D': '#6387A6', 'PlantSeg': '#C5E2E6',
    'Cellpose': '#8bc34a', 'Omnipose': '#c2b7f3', 'CellSeg3D': '#5ba8ec',
    'Cellpose-SAM': '#7FC7AF', 'uSAM': '#9e6b4d', 'Ilastik': 'c',
    'CellProfiler': '#cc5618', 'Swin-UNETR': '#ff5e6e', 'MedSAM': '#ffdd2e',
    'Mesmer': '#f9a825', 'Cellpose-2D': '#f3d266', '3DCellScope': '#806da5',
    'Med-SA': '#cc5618', 'u-Segment3D': '#6e597f', 'CellSAM': '#0055b3',
}
ERROR_COLORS = {'MNR': '#FF7F0F', 'USR': '#2BA02B', 'OSR': '#287CFF'}


In [ ]:
# Validate the input table and retain the selected methods.
if not DATA_PATH.is_file():
    raise FileNotFoundError(f'CSV not found; update DATA_PATH: {DATA_PATH}')
if not isinstance(TOP_K, int) or TOP_K < 1:
    raise ValueError('TOP_K must be a positive integer.')
SAVE_PATH.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(DATA_PATH)
if 'Method' not in df.columns:
    raise ValueError('Missing required column: Method')
if AP_COLUMN not in df.columns:
    raise ValueError(f'Missing AP column: {AP_COLUMN}. Set AP_COLUMN explicitly.')
missing_columns = [column for column in ['Method'] + RI_METRICS if column not in df.columns]
if missing_columns:
    raise ValueError(f'Missing required columns: {missing_columns}')

# Treat nonnumeric and infinite values as missing.
for metric in RI_METRICS:
    df[metric] = pd.to_numeric(df[metric], errors='coerce').replace([np.inf, -np.inf], np.nan)
observed_methods = df['Method'].dropna().unique().tolist()
current_methods = [m for m in METHOD_LIST if m in observed_methods]
missing_methods = [m for m in METHOD_LIST if m not in observed_methods]
ignored_methods = [m for m in observed_methods if m not in METHOD_LIST]
if missing_methods:
    print('Methods in METHOD_LIST but not found:', missing_methods)
if ignored_methods:
    print('Methods excluded by METHOD_LIST:', ignored_methods)
if not current_methods:
    raise ValueError('No methods in METHOD_LIST were found in the CSV.')
df = df[df['Method'].isin(current_methods)].copy()
print(f'Rows retained: {len(df)}; methods: {len(current_methods)}; AP source: {AP_COLUMN}')
display(df[['Method'] + RI_METRICS].head())


In [ ]:
# Compute method-level means and sample SDs; use zero SD for single samples.
method_mean = df.groupby('Method')[RI_METRICS].mean().reindex(current_methods)
method_std = df.groupby('Method')[RI_METRICS].std().reindex(current_methods).fillna(0)

# RI requires a valid mean for every metric.
complete_mean = method_mean.dropna(subset=RI_METRICS).copy()
excluded_ri = method_mean.index[method_mean.isna().any(axis=1)].tolist()
if excluded_ri:
    print('Excluded from RI because a metric mean is missing:', excluded_ri)
if complete_mean.empty:
    raise ValueError('No complete methods are available for RI calculation.')

# Better values receive larger ranks; ties use average ranks.
ranklist = pd.DataFrame(index=complete_mean.index)
for metric in RI_METRICS:
    ranklist[f'mean_{metric}'] = complete_mean[metric]
    ranklist[f'rank_{metric}'] = complete_mean[metric].rank(
        ascending=(metric in HIGHER_IS_BETTER), method='average'
    )
# Sum the six ranks with equal weights; higher RI is better.
ranklist['RI'] = ranklist[[f'rank_{m}' for m in RI_METRICS]].sum(axis=1)
ranklist = ranklist.sort_values('RI', ascending=False, kind='stable')
ranklist.index.name = 'Method'
ri_method_order = ranklist.index.tolist()
method_mean_ri = method_mean.reindex(ri_method_order)
method_std_ri = method_std.reindex(ri_method_order)
bar_y = np.arange(len(ri_method_order))
print('Bar-chart order (higher RI first):', ri_method_order)
display(method_mean_ri.round(4))


In [ ]:
# Shared helpers for styling, axis limits, and figure export.
def new_axes(title, scatter=False):
    size = SCATTER_FIGSIZE if scatter else FIGSIZE
    fig, ax = plt.subplots(figsize=size, layout='constrained')
    ax.set_box_aspect(1)
    ax.set_title(title, fontsize=12, pad=12)
    return fig, ax

def style_bar_axes(ax, xlabel):
    ax.set_yticks(bar_y)
    ax.set_yticklabels(ri_method_order, fontsize=8.5)
    ax.invert_yaxis()
    ax.set_xlabel(xlabel, fontsize=11)
    ax.set_ylabel('')
    ax.tick_params(axis='x', labelsize=9)
    ax.set_axisbelow(True)
    ax.grid(axis='x', linestyle='--', linewidth=0.7, alpha=0.30, zorder=0)
    sns.despine(ax=ax)

# Expand limits to include the full mean +/- SD range.
def set_error_limits(ax, means, stds):
    means, stds = np.asarray(means, float), np.asarray(stds, float)
    valid = np.isfinite(means) & np.isfinite(stds)
    if not valid.any():
        ax.set_xlim(0, 1)
        return
    left = min(0.0, float(np.min(means[valid] - stds[valid])))
    right = float(np.max(means[valid] + stds[valid]))
    padding = max(right - left, 0.1) * 0.07
    ax.set_xlim(left, max(right + padding, left + 0.1))

def save_and_show(fig, stem):
    for fmt in EXPORT_FORMATS:
        path = SAVE_PATH / f'{stem}.{fmt}'
        fig.savefig(path, dpi=DPI, bbox_inches='tight')
        print('Saved:', path)
    if SHOW_FIGURES:
        plt.show()
    plt.close(fig)

def get_top_k(metric):
    values = method_mean[metric].dropna().sort_values(
        ascending=(metric in LOWER_IS_BETTER), kind='stable'
    )
    return values.head(TOP_K)


In [ ]:
# Figure 1: FR mean +/- SD, with methods ordered by RI.
fig_FR, ax_FR = new_axes('FR')
FR_means = method_mean_ri['FR'].to_numpy()
FR_stds = method_std_ri['FR'].to_numpy()
bar_colors = [METHOD_COLORS.get(m, '#808080') for m in ri_method_order]
ax_FR.barh(bar_y, FR_means, height=0.68, color=bar_colors,
            edgecolor=bar_colors, linewidth=0.7, alpha=0.90, zorder=3)
ax_FR.errorbar(FR_means, bar_y, xerr=FR_stds, fmt='none',
                ecolor='#4A4A4A', elinewidth=1.0, capsize=3, capthick=0.9, zorder=4)
style_bar_axes(ax_FR, 'FR')
set_error_limits(ax_FR, FR_means, FR_stds)
save_and_show(fig_FR, '01_FR_mean_SD')


In [ ]:
# Use the same complete samples for stacked means and total-error SD.
error_sample_df = df.dropna(subset=['MNR', 'USR', 'OSR']).copy()
# Sum components per sample before estimating SD across samples.
error_sample_df['Error_rate'] = error_sample_df[['MNR', 'USR', 'OSR']].sum(axis=1)
error_components = error_sample_df.groupby('Method')[['MNR', 'USR', 'OSR']].mean().reindex(ri_method_order)
error_stats = error_sample_df.groupby('Method')['Error_rate'].agg(['mean', 'std']).reindex(ri_method_order)
error_stats['std'] = error_stats['std'].fillna(0)
missing_error_methods = error_stats.index[error_stats['mean'].isna()].tolist()
if missing_error_methods:
    print('No complete error-rate samples for:', missing_error_methods)

# Figure 2: stack MNR, USR, and OSR; add one SD bar at the total.
fig_error, ax_error = new_axes('Error rate')
left = np.zeros(len(ri_method_order))
for metric in ['MNR', 'USR', 'OSR']:
    values = error_components[metric].to_numpy()
    ax_error.barh(bar_y, values, height=0.68, left=left,
                  color=ERROR_COLORS[metric], edgecolor='white', linewidth=0.6, label=metric, zorder=3)
    left = left + values
total_error = error_stats['mean'].to_numpy()
total_error_std = error_stats['std'].to_numpy()
valid = np.isfinite(total_error)
ax_error.errorbar(total_error[valid], bar_y[valid], xerr=total_error_std[valid],
                  fmt='none', ecolor='#111111', elinewidth=1.1, capsize=3, capthick=1.0, zorder=6)
style_bar_axes(ax_error, 'Error rate')
set_error_limits(ax_error, total_error, total_error_std)
ax_error.legend(handles=[Patch(facecolor=ERROR_COLORS[m], edgecolor='none', label=m)
                        for m in ['MNR', 'USR', 'OSR']],
                loc='upper right', frameon=False, fontsize=8.5)
save_and_show(fig_error, '02_Error_rate_stacked')


In [ ]:
# Figure 3: method-level mIoU and AP means with SD on both axes.
fig_scatter, ax_scatter = new_axes('mIoU × AP', scatter=True)
top_iou = get_top_k('IoU')
top_ap = get_top_k('AP')
methods_to_label = set(top_iou.index) | set(top_ap.index)
for method in current_methods:
    mean_x, mean_y = method_mean.loc[method, ['IoU', 'AP']]
    if pd.isna(mean_x) or pd.isna(mean_y):
        continue
    std_x, std_y = method_std.loc[method, ['IoU', 'AP']]
    color = METHOD_COLORS.get(method, '#808080')
    ax_scatter.errorbar(mean_x, mean_y, xerr=std_x, yerr=std_y, fmt='o', markersize=6.5,
                        markerfacecolor=color, markeredgecolor=color, markeredgewidth=0.5,
                        ecolor=color, elinewidth=1.0, capsize=3, capthick=0.9,
                        alpha=0.90, zorder=3, label=method)
    if LABEL_TOP_METHODS and method in methods_to_label:
        ax_scatter.annotate(method, (mean_x, mean_y), xytext=(3, 4),
                            textcoords='offset points', fontsize=8, color='#404040', clip_on=False)
# Reference lines mark the TOP_K-th best mean for each metric.
if not top_iou.empty:
    ax_scatter.axvline(top_iou.iloc[-1], color='#7f7f7f', linestyle=(0, (6, 4)),
                       linewidth=1.25, alpha=0.85, zorder=1)
if not top_ap.empty:
    ax_scatter.axhline(top_ap.iloc[-1], color='#7f7f7f', linestyle=(0, (6, 4)),
                       linewidth=1.25, alpha=0.85, zorder=1)
ax_scatter.set_xlabel('mIoU', fontsize=11)
ax_scatter.set_ylabel('AP', fontsize=11)
if SCATTER_XLIM is not None:
    ax_scatter.set_xlim(*SCATTER_XLIM)
    if tuple(SCATTER_XLIM) == (0.49, 1.01):
        ax_scatter.set_xticks(np.arange(0.50, 1.01, 0.1))
ax_scatter.tick_params(axis='both', labelsize=9)
ax_scatter.set_axisbelow(True)
ax_scatter.grid(True, linestyle='--', linewidth=0.7, alpha=0.30, zorder=0)
ax_scatter.margins(y=0.10)
sns.despine(ax=ax_scatter)
if SHOW_SCATTER_LEGEND:
    ax_scatter.legend(loc='upper left', bbox_to_anchor=(1.02, 1),
                      frameon=False, fontsize=8, borderaxespad=0)
save_and_show(fig_scatter, '03_mIoU_AP_scatter')


In [ ]:
# Export the ranking table, then draw RI as the final figure.
rank_output_path = SAVE_PATH / 'global_RI_6metrics_all_methods.csv'
ranklist.reset_index().to_csv(rank_output_path, index=False)
print('Saved RI table:', rank_output_path)
display(ranklist.round(4))

fig_ri, ax_ri = new_axes('Comprehensive RI ranking')
ri_values = ranklist['RI'].reindex(ri_method_order).to_numpy()
ri_colors = [METHOD_COLORS.get(m, '#808080') for m in ri_method_order]
ax_ri.barh(bar_y, ri_values, height=0.68, color=ri_colors, edgecolor=ri_colors,
           linewidth=0.7, alpha=0.90, zorder=3)
style_bar_axes(ax_ri, 'RI')
ax_ri.set_xlim(0, max(float(np.nanmax(ri_values)) * 1.07, 1.0))
save_and_show(fig_ri, '04_RI_ranking')
